# Ventas de videojuegos

Cinco gráficos con datos de Kaggle (`gregorut/videogamesales`). Primero Matplotlib, luego los mismos con Seaborn, y al final la página con Streamlit.


## 1. Descarga del conjunto de datos desde Kaggle

Para iniciar el análisis, utilizaremos la API oficial de Kaggle para descargar el archivo de ventas de videojuegos directamente en el entorno local de Google Colab.

En [kaggle.com/settings/api](https://www.kaggle.com/settings/api), pulsa **Create Legacy API Key** y sube aquí el `kaggle.json`.


In [1]:
!pip install -q kaggle seaborn streamlit


In [ ]:
from google.colab import files
from pathlib import Path
import os

uploaded = files.upload()

destino = Path.home() / ".kaggle"
destino.mkdir(parents=True, exist_ok=True)
nombre = next(iter(uploaded))
texto = uploaded[nombre].decode("utf-8").strip()

if texto.startswith("{") and "key" in texto:
    ruta = destino / "kaggle.json"
    ruta.write_bytes(uploaded[nombre])
elif texto.startswith("KGAT_"):
    ruta = destino / "access_token"
    ruta.write_text(texto + "\n", encoding="utf-8")
else:
    raise SystemExit("Sube el kaggle.json de Kaggle.")

os.chmod(ruta, 0o600)


In [ ]:
!mkdir -p data
!kaggle datasets download -d gregorut/videogamesales -p data --unzip


## 2. Carga y preprocesamiento de los datos

En esta sección importamos las librerías necesarias (`pandas`, `matplotlib` y `seaborn`) y limpiamos el conjunto de datos. Filtramos los registros sin año de lanzamiento o género específico, acotamos los años entre 1980 y 2016 para evitar datos incompletos, y traducimos los nombres de los géneros al español para facilitar su lectura.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

SURFACE = "#fcfcfb"
TEXT, TEXT2, GRID = "#0b0b0b", "#52514e", "#e3e2de"
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT2, "xtick.color": TEXT2, "ytick.color": TEXT2,
    "text.color": TEXT, "axes.titlelocation": "left", "axes.titleweight": "semibold",
    "axes.titlesize": 12, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 1,
    "lines.linewidth": 2, "lines.solid_capstyle": "round", "legend.frameon": False,
    "font.size": 10, "figure.dpi": 110,
})

def xgrid(ax):
    ax.grid(False)
    ax.grid(True, axis="x")

def ygrid(ax):
    ax.grid(False)
    ax.grid(True, axis="y")

df = pd.read_csv("data/vgsales.csv")
df = df.dropna(subset=["Year", "Genre"]).copy()
df["Year"] = df["Year"].astype(int)
df = df[(df["Year"] >= 1980) & (df["Year"] <= 2016)]

GENEROS = {
    "Action": "Acción",
    "Sports": "Deportes",
    "Shooter": "Disparos",
    "Role-Playing": "Rol",
    "Platform": "Plataformas",
    "Misc": "Otros",
    "Racing": "Carreras",
    "Fighting": "Lucha",
    "Simulation": "Simulación",
    "Puzzle": "Puzles",
    "Adventure": "Aventura",
    "Strategy": "Estrategia",
}
df["Genero"] = df["Genre"].map(GENEROS).fillna(df["Genre"])

por_genero = (
    df.groupby("Genero", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Global_Sales", ascending=False)
)
por_anio = df.groupby("Year", as_index=False)["Global_Sales"].sum()
top = df.nlargest(200, "Global_Sales")
regiones = pd.Series({
    "Norteamérica": df["NA_Sales"].sum(),
    "Europa": df["EU_Sales"].sum(),
    "Japón": df["JP_Sales"].sum(),
    "Resto": df["Other_Sales"].sum(),
})
df.head()


## Matplotlib


### Barras


Visualización del acumulado de ventas globales en millones de copias para cada uno de los géneros de videojuegos mediante un gráfico de barras horizontales.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
barras = ax.barh(por_genero["Genero"], por_genero["Global_Sales"], color=PAL[0], height=0.62)
ax.invert_yaxis()
ax.bar_label(barras, fmt="%.0f", padding=3, color=TEXT, fontsize=8)
ax.set_xlim(0, por_genero["Global_Sales"].max() * 1.16)
ax.set_xlabel("Ventas globales (millones de copias)")
xgrid(ax)
ax.set_title("Ventas globales por género")
fig.tight_layout()
plt.show()


### Línea


Evolución histórica de las ventas globales anuales agregadas para observar la tendencia del mercado a lo largo del tiempo.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(por_anio["Year"], por_anio["Global_Sales"], color=PAL[0])
ultimo_x = por_anio["Year"].iloc[-1]
ultimo_y = por_anio["Global_Sales"].iloc[-1]
ax.plot(ultimo_x, ultimo_y, "o", color=PAL[0], markersize=8, markeredgecolor=SURFACE, markeredgewidth=2)
ax.annotate(f"{ultimo_y:.0f}", (ultimo_x, ultimo_y), xytext=(8, 0), textcoords="offset points", va="center", color=TEXT)
ax.set_xticks(por_anio["Year"][::5])
ax.set_xlabel("Año")
ax.set_ylabel("Ventas globales (millones)")
ygrid(ax)
ax.set_title("Ventas globales por año")
fig.tight_layout()
plt.show()


### Dispersión


Análisis de correlación para evaluar el comportamiento de los 200 juegos más vendidos entre el mercado norteamericano (NA) y el mercado europeo (EU).

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.scatter(top["NA_Sales"], top["EU_Sales"], s=72, color=PAL[0], edgecolor=SURFACE, linewidth=1.5)
ax.set_xlabel("Ventas en Norteamérica (millones)")
ax.set_ylabel("Ventas en Europa (millones)")
ax.set_title("Norteamérica frente a Europa")
fig.tight_layout()
plt.show()


### Histograma


Distribución del volumen de lanzamientos de videojuegos registrados en el conjunto de datos por año.

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.hist(df["Year"], bins=18, color=PAL[0], edgecolor=SURFACE, linewidth=1.5)
ax.set_xlabel("Año de lanzamiento")
ax.set_ylabel("Juegos")
ygrid(ax)
ax.set_title("Lanzamientos por año")
fig.tight_layout()
plt.show()


### Tarta


Distribución porcentual de las ventas acumuladas por regiones geográficas principales (Norteamérica, Europa, Japón y Resto del mundo).

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.pie(
    regiones.values,
    labels=[f"{nombre}\n{valor:.0f} M" for nombre, valor in regiones.items()],
    colors=PAL[:4],
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": SURFACE, "linewidth": 2},
    textprops={"color": TEXT},
)
ax.axis("equal")
ax.grid(False)
ax.set_title("Ventas por región")
fig.tight_layout()
plt.show()


## Seaborn


## 3. Réplica de los gráficos con Seaborn

A continuación, replicamos la misma estructura de análisis utilizando la librería Seaborn para aprovechar su estilo visual predeterminado y capacidades de análisis estadístico rápido.

In [ ]:
sns.set_theme(style="whitegrid", palette=PAL)
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT2, "xtick.color": TEXT2, "ytick.color": TEXT2,
    "text.color": TEXT, "axes.titlelocation": "left", "axes.titleweight": "semibold",
    "axes.titlesize": 12, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "font.size": 10, "figure.dpi": 110,
})


### Barras


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
sns.barplot(data=por_genero, y="Genero", x="Global_Sales", color=PAL[0], order=por_genero["Genero"], ax=ax)
ax.set_xlim(0, por_genero["Global_Sales"].max() * 1.12)
ax.set_xlabel("Ventas globales (millones de copias)")
ax.set_ylabel("")
xgrid(ax)
ax.set_title("Ventas globales por género")
fig.tight_layout()
plt.show()


### Línea


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
sns.lineplot(data=por_anio, x="Year", y="Global_Sales", color=PAL[0], ax=ax)
ultimo_x = por_anio["Year"].iloc[-1]
ultimo_y = por_anio["Global_Sales"].iloc[-1]
ax.plot(ultimo_x, ultimo_y, "o", color=PAL[0], markersize=8, markeredgecolor=SURFACE, markeredgewidth=2)
ax.annotate(f"{ultimo_y:.0f}", (ultimo_x, ultimo_y), xytext=(8, 0), textcoords="offset points", va="center", color=TEXT)
ax.set_xticks(por_anio["Year"][::5])
ax.set_xlabel("Año")
ax.set_ylabel("Ventas globales (millones)")
ygrid(ax)
ax.set_title("Ventas globales por año")
fig.tight_layout()
plt.show()


### Dispersión


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.4))
sns.scatterplot(
    data=top, x="NA_Sales", y="EU_Sales", color=PAL[0],
    s=72, edgecolor=SURFACE, linewidth=1.5, ax=ax, legend=False,
)
ax.set_xlabel("Ventas en Norteamérica (millones)")
ax.set_ylabel("Ventas en Europa (millones)")
ax.set_title("Norteamérica frente a Europa")
fig.tight_layout()
plt.show()


### Histograma


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.8))
sns.histplot(data=df, x="Year", bins=18, color=PAL[0], edgecolor=SURFACE, linewidth=1.5, ax=ax)
ax.set_xlabel("Año de lanzamiento")
ax.set_ylabel("Juegos")
ygrid(ax)
ax.set_title("Lanzamientos por año")
fig.tight_layout()
plt.show()


### Tarta


In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.pie(
    regiones.values,
    labels=[f"{nombre}\n{valor:.0f} M" for nombre, valor in regiones.items()],
    colors=sns.color_palette(PAL[:4]),
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": SURFACE, "linewidth": 2},
    textprops={"color": TEXT},
)
ax.axis("equal")
ax.grid(False)
ax.set_title("Ventas por región")
fig.tight_layout()
plt.show()


## 4. Creación y Despliegue de la Aplicación con Streamlit

Para interactuar con el análisis de forma dinámica, crearemos una interfaz web con Streamlit que recopila los gráficos creados anteriormente.

Ejecuta la celda de abajo para iniciar el servidor de Streamlit y generar tu enlace seguro de acceso.

In [ ]:
%%writefile app.py
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Ventas de videojuegos", layout="centered")

SURFACE = "#fcfcfb"
TEXT = "#0b0b0b"
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
AZUL = "#2a78d6"

GENEROS = {
    "Action": "Acción",
    "Sports": "Deportes",
    "Shooter": "Disparos",
    "Role-Playing": "Rol",
    "Platform": "Plataformas",
    "Misc": "Otros",
    "Racing": "Carreras",
    "Fighting": "Lucha",
    "Simulation": "Simulación",
    "Puzzle": "Puzles",
    "Adventure": "Aventura",
    "Strategy": "Estrategia",
}

BARRAS = """
st.bar_chart(
    por_genero,
    x="Genero",
    y="Global_Sales",
    x_label="Género",
    y_label="Ventas globales (millones de copias)",
    color=AZUL,
    horizontal=True,
    sort=False,
)
"""

LINEA = """
st.line_chart(
    por_anio,
    x="Year",
    y="Global_Sales",
    x_label="Año",
    y_label="Ventas globales (millones)",
    color=AZUL,
)
"""

DISPERSION = """
st.scatter_chart(
    top,
    x="NA_Sales",
    y="EU_Sales",
    x_label="Ventas en Norteamérica (millones)",
    y_label="Ventas en Europa (millones)",
    color=AZUL,
)
"""

HISTOGRAMA = """
lanzamientos = (
    df["Year"].value_counts().sort_index()
    .rename_axis("Year")
    .reset_index(name="Juegos")
)
st.bar_chart(
    lanzamientos,
    x="Year",
    y="Juegos",
    x_label="Año de lanzamiento",
    y_label="Juegos",
    color=AZUL,
    sort=False,
)
"""

TARTA = """
fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.pie(
    regiones.values,
    labels=[f"{nombre}\\n{valor:.0f} M" for nombre, valor in regiones.items()],
    colors=PAL,
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": SURFACE, "linewidth": 2},
    textprops={"color": TEXT},
)
ax.axis("equal")
ax.set_title("Ventas por región")
fig.tight_layout()
st.pyplot(fig)
"""

st.title("Ventas de videojuegos")
st.write("Los mismos cinco gráficos, dibujados con Streamlit.")
st.markdown(
    "<style>[data-testid='stCode'] code { white-space: pre-wrap !important; word-break: break-word; }</style>",
    unsafe_allow_html=True,
)

if not Path("data/vgsales.csv").exists():
    st.error("Falta data/vgsales.csv. Ejecuta antes la descarga de Kaggle.")
    st.stop()

df = pd.read_csv("data/vgsales.csv")
df = df.dropna(subset=["Year", "Genre"]).copy()
df["Year"] = df["Year"].astype(int)
df = df[(df["Year"] >= 1980) & (df["Year"] <= 2016)]
df["Genero"] = df["Genre"].map(GENEROS).fillna(df["Genre"])

por_genero = (
    df.groupby("Genero", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Global_Sales", ascending=False)
)
por_anio = df.groupby("Year", as_index=False)["Global_Sales"].sum()
top = df.nlargest(200, "Global_Sales")
regiones = pd.Series({
    "Norteamérica": df["NA_Sales"].sum(),
    "Europa": df["EU_Sales"].sum(),
    "Japón": df["JP_Sales"].sum(),
    "Resto": df["Other_Sales"].sum(),
})

def seccion(nombre, que_es, codigo):
    st.header(nombre)
    st.write(que_es)
    exec(codigo, {
        "st": st, "plt": plt, "df": df, "AZUL": AZUL, "PAL": PAL, "SURFACE": SURFACE, "TEXT": TEXT,
        "por_genero": por_genero, "por_anio": por_anio, "top": top, "regiones": regiones,
    })
    st.code(codigo.strip(), language="python")

seccion("Barras", "Comparación de ventas por género, con st.bar_chart.", BARRAS)
seccion("Línea", "Ventas globales cada año, con st.line_chart.", LINEA)
seccion("Dispersión", "Norteamérica frente a Europa, con st.scatter_chart.", DISPERSION)
seccion("Histograma", "Juegos lanzados por año. Streamlit no tiene histograma: el recuento se dibuja con st.bar_chart.", HISTOGRAMA)
seccion("Tarta", "Reparto por región. Streamlit no tiene tarta: el gráfico se muestra con st.pyplot.", TARTA)

In [ ]:
import subprocess
import time
from google.colab import output

# 1. Asegurar que no hay procesos de Streamlit previos activos
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
time.sleep(2)

# 2. Iniciar Streamlit en segundo plano apuntando al puerto local 8501
print("Iniciando el servidor de Streamlit...")
log_st = open("streamlit.log", "w")
subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false"
    ],
    stdout=log_st,
    stderr=subprocess.STDOUT,
)
time.sleep(4)

# 3. Generar el enlace seguro oficial de Google Colab de forma nativa
enlace_seguro = output.eval_js("google.colab.kernel.proxyPort(8501)")
print(f"\n👉 La aplicación está lista. Accede haciendo clic aquí:\n{enlace_seguro}")